# Tutorial 6: Train NicheTrans* on STARmap PLUS data

In [1]:
import os, time, datetime, warnings

import torch
import torch.nn as nn
from torch.optim import lr_scheduler

from model.nicheTrans_ct import *
from datasets.data_manager_STARmap_PLUS import AD_Mouse

from utils.utils import *
from utils.utils_training_STARmap_PLUS import train, test
from utils.utils_dataloader import *

warnings.filterwarnings("ignore")

### Initialize the args and fix seeds

In [2]:
%run ./args/args_STARmap_PLUS.py
args = args

set_seed(args.seed)
os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu_devices

print("==========\nArgs:{}\n==========".format(args))

Args:Namespace(noise_rate=0.5, dropout_rate=0.25, n_top_genes=2000, workers=4, AD_adata_path='/mnt/datadisk0/Processed_DATA/2023_nn_AD_mouse/AD_model_adata_protein', Wild_type_adata_path='/mnt/datadisk0/Processed_DATA/2023_nn_AD_mouse/wild_type_adata_protein', cell_type_visualize=False, cell_type_visualization_dir=None, cell_type_visualization_dpi=150, max_epoch=20, stepsize=20, train_batch=128, test_batch=32, optimizer='adam', lr=0.0001, gamma=0.1, weight_decay=0.0005, seed=1, save_dir='./log', eval_step=1, gpu_devices='0')


### Initialize dataloaders and NicheTrans

In [3]:
# create the dataloaders
dataset = AD_Mouse(AD_adata_path=args.AD_adata_path, Wild_type_adata_path=args.Wild_type_adata_path, n_top_genes=args.n_top_genes,cell_type_visualize=True)
trainloader, testloader, _ = ad_mouse_dataloader(args, dataset)

# create the model
source_dimension, target_dimension = dataset.rna_length, dataset.target_length
n_spot_types = getattr(dataset, 'n_spot_types', 1)
model_kwargs = dict(
    source_length=source_dimension,
    target_length=target_dimension,
    noise_rate=args.noise_rate,
    dropout_rate=args.dropout_rate,
    n_spot_types=n_spot_types,
    n_cell_types=getattr(dataset, 'n_cell_types', n_spot_types),
)
model = NicheTrans_ct(**model_kwargs)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model = model.to(device)


  Using provided cell-type annotation "ct_top" with 13 global cell types


  Scanpy visualizations saved to: scanpy_cell_type_visualizations
------Calculating spatial graph...


The graph contains 124464 edges, 10372 cells.
12.0000 neighbors per cell on average.


------Calculating spatial graph...


The graph contains 115608 edges, 9634 cells.
12.0000 neighbors per cell on average.


------Calculating spatial graph...


The graph contains 96408 edges, 8034 cells.
12.0000 neighbors per cell on average.


=> AD Mouse loaded
Dataset statistics:
  ------------------------------
  subset   | # num | 
  ------------------------------
  train    |  10372 spots, 894.0 positive tao, 291.0 positive plaque 
  test     |   9634 spots, 620.0 positive tao, 195.0 positive plaque 
  ------------------------------
  Global cell-type source: annotation
  Global cell types (13): ['Astro', 'CA1', 'CA2', 'CA3', 'CTX-Ex', 'DG', 'Endo', 'Inh', 'LHb', 'Micro', 'OPC', 'Oligo', 'SMC']
  Cell-type visualization slices: ['13months-disease-replicate_1_random.h5ad', '13months-disease-replicate_2_random.h5ad', 'spatial_13months-control-replicate_1.h5ad']


### Initialize loss function (criterion) and optimizer

In [4]:
criterion = nn.BCELoss()

if args.optimizer == 'adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
elif args.optimizer == 'SGD':
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr)
else:
    print('unexpected optimizer')

if args.stepsize > 0:
    scheduler = lr_scheduler.StepLR(optimizer, step_size=args.stepsize, gamma=args.gamma)

### Model training and testing

In [5]:
start_time = time.time()

for epoch in range(args.max_epoch):
    last_epoch = epoch + 1 == args.max_epoch

    print("==> Epoch {}/{}".format(epoch+1, args.max_epoch))
    
    ################
    train(model, criterion, optimizer, trainloader, ct_information=True, device=device)
    if args.stepsize > 0: scheduler.step()
    ################

pearson = test(model, testloader, ct_information=True, device=device)
torch.save(model.state_dict(), 'NicheTrans_*_STARmap_PLUS.pth')

elapsed = round(time.time() - start_time)
elapsed = str(datetime.timedelta(seconds=elapsed))
print("Finished. Total elapsed time (h:m:s): {}".format(elapsed))

==> Epoch 1/20


Batch 81/81	 Loss 0.515318 (0.588264)
==> Epoch 2/20


Batch 81/81	 Loss 0.382142 (0.446976)
==> Epoch 3/20


Batch 81/81	 Loss 0.322357 (0.355444)
==> Epoch 4/20


Batch 81/81	 Loss 0.259365 (0.291052)
==> Epoch 5/20


...

Batch 81/81	 Loss 0.076015 (0.111190)
==> Epoch 18/20


Batch 81/81	 Loss 0.094680 (0.109410)
==> Epoch 19/20


Batch 81/81	 Loss 0.104982 (0.106893)
==> Epoch 20/20


Batch 81/81	 Loss 0.076192 (0.101878)


tau AUC: 0.8592161297479907, tau sensitivity 0.6080645161290322, tay specificity 0.9120257377412914
Aβ AUC: 0.919360753665235, Aβ sensitivity 0.5282051282051282, Aβ specificity 0.9852738637567539
Finished. Total elapsed time (h:m:s): 0:03:03
